In [6]:
# Build a SQLite DB of IPL 2021-2024 from the kaggle IPL complete dataset.Expects matches.csv and deliveries.csv in the same folder.
# Produce: ipl_2021_2024.db with two linked tables: matches,deliveries

import sqlite3
import pandas as pd

MATCHES_CSV = "data/matches.csv"
DELIVERIES_CSV = "data/deliveries.csv"
DB_PATH = "ipl_2021_2024.db"
YEARS =  {2021,2022,2023,2024}

# 1. Load matches
matches = pd.read_csv(MATCHES_CSV)

# The 'session' column in this dataset is messy: some rows are "2021",
# others are like "2020/21" or "2009/10".Safest filter is the match DATE,
# not the season string. Parse date and extract the year.
matches["date"] = pd.to_datetime(matches["date"],errors="coerce",dayfirst=False)


# a few rows may fail to parse depending on farmat; try dayfirst as fallback
if matches["date"].isna().any():
    fallback = pd.to_datetime(
        matches.loc[matches["date"].isna(),"date"] if False else
        pd.read_csv(MATCHES_CSV)['date'],
        erros="coerce",dayfirst=True
    )
    matches["date"] = matches["date"].fillna(fallback)

matches["year"] = matches["date"].dt.year

matches_filtered = matches[matches["year"].isin(YEARS)].copy()

# The match identifier column is named 'id' in this dataset.
match_ids = set(matches_filtered["id"].unique())
print(f"Matches in 2021-2024: {len(matches_filtered)}")

Matches in 2021-2024: 279
